In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import snowflake.connector

# If using local Jupyter, uncomment and fill this:
# conn = snowflake.connector.connect(
#     user='YOUR_USER',
#     password='YOUR_PASSWORD',
#     account='YOUR_ACCOUNT',
#     warehouse='PORTFOLIO_WH',
#     database='AIRBNB_PORTFOLIO',
#     schema='ANALYTICS'
# )

# If using Snowflake Notebooks, use the active session:
session = get_active_session() 

# 1. Load the data into a Snowpark DataFrame
df = session.table("AIRBNB_PORTFOLIO.ANALYTICS.INT_LISTINGS_ENRICHED")

# 2. Filter to only the columns we need for ML
# We exclude ID, Names, URLs, and things that leak the target (like estimated_revenue)
ml_df = df.select(
    "PRICE_USD",
    "ACCOMMODATES",
    "BEDROOMS",
    "BEDS",
    "BATHROOMS",
    "ROOM_TYPE",
    "BOROUGH",
    "NEIGHBOURHOOD_MEDIAN_PRICE",
    "REVIEW_SCORES_RATING",
    "MINIMUM_NIGHTS",
    "AVAILABILITY_365",
    "HOST_LISTING_COUNT"
)

# 3. Drop rows where the target (price) is null, and filter out extreme outliers (keep our cleaning logic)
ml_df = ml_df.filter(
    (ml_df["PRICE_USD"].isNotNull()) & 
    (ml_df["PRICE_USD"] < 2000)
)

# 4. Convert to a pandas DataFrame for Scikit-Learn/XGBoost
# (Since our dataset is only ~20k rows, it fits comfortably in memory)
pdf = ml_df.to_pandas()

print(f"Dataset shape: {pdf.shape}")
print(pdf.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Define X (features) and y (target)
X = pdf.drop(columns=["PRICE_USD"])
y = pdf["PRICE_USD"]

# 2. Identify numeric and categorical columns
numeric_features = ["ACCOMMODATES", "BEDROOMS", "BEDS", "BATHROOMS", 
                    "NEIGHBOURHOOD_MEDIAN_PRICE", "REVIEW_SCORES_RATING", 
                    "MINIMUM_NIGHTS", "AVAILABILITY_365", "HOST_LISTING_COUNT"]
categorical_features = ["ROOM_TYPE", "BOROUGH"]

# 3. Build preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Fill missing bedrooms with median
    ('scaler', StandardScaler())                   # Scale numeric features
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # Convert categories to 0/1
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 4. Create the full pipeline with a Random Forest model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# 5. Train/Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Train the model
model.fit(X_train, y_train)

# 7. Evaluate
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"R-squared (R2): {r2:.4f}")

In [ ]:
from snowflake.ml.registry import Registry
import snowflake.snowpark as snowpark

# Get the active Snowpark session (if you are in Snowflake Notebooks, this is already active)
session = get_active_session()

# Initialize the registry targeting your ANALYTICS schema
reg = Registry(session=session, database_name="AIRBNB_PORTFOLIO", schema_name="ANALYTICS")

# Register the model
mv = reg.log_model(
    model=model,
    model_name="AIRBNB_PRICE_PREDICTOR",
    version_name="v1",
    sample_input_data=X_train, # Important: Snowflake uses this to infer the model signature
    comment="Random Forest model predicting nightly Airbnb price in NYC. R2=0.65, MAE=$73.75."
)

print("Model registered successfully!")
print(f"Model Name: {mv.model_name}")
print(f"Version: {mv.version_name}")